In [23]:
# Chunking using smart splitter

import os
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Loading our document
with open("my_knowledge.txt") as f:
    knowledge_text = f.read()

# text splitter smartly splits the text into semantically meaningful chunks as much as possible
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 180,   # Max size of chunk
    chunk_overlap = 20, # Overlap to maintain context
    length_function = len, # Function to calculate length of text
)

# creating the chunks
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n {chunk}\n")



We have 5 chunks
--- Chunk 1 ---
 Company Policy Manual:

--- Chunk 2 ---
 - WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote

--- Chunk 3 ---
 are optional remote days.

--- Chunk 4 ---
 - PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.

--- Chunk 5 ---
 - Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.



In [24]:
# Next step is to create embeddings for these chunks i.e converting them into numbers(vectors)
# We will use sentence-transformer model which is great at understanding the meaning of a sentence
import warnings
warnings.filterwarnings("ignore")

from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer('all-MiniLM-L6-v2') # loading the model

# Embed all the chunks
# This takes time depending on the number of chunks and the model used
chunk_embeddings = model.encode(chunks)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1643.92it/s]


Chunk embeddings shape: (5, 384)


In [25]:
# Creating a vector database using FAISS to store the embeddings
import faiss
import numpy as np

# Getting the dimension of our vectors
d = chunk_embeddings.shape[1]  # dimension of the vectors

# 1. Create a FAISS index
# # IndexFlatL2 is the simplest, most basic index. It calculates the exact distance (L2 distance) between our query and all vectors.

index = faiss.IndexFlatL2(d)  # L2 distance index

# 2. Add our chunk embeddings to the index 
# We must to float32 for FAISS
index.add(np.array(chunk_embeddings, dtype=np.float32))

print(f"Number of vectors in the index: {index.ntotal}")

Number of vectors in the index: 5


In [29]:
# Next step is to retrieve, augment and generate
# The full pipeline is as follows:

from transformers import pipeline
# 1. Load a question-answering model or text generation model
generator = pipeline('text-generation', model='Qwen/Qwen3-4B-Instruct-2507')  # You can choose a different model if you want

# Our RAG pipeline function
def answer_question(query):
    # 1. RETRIEVE
    # Embed the users query
    query_embediing = model.encode([query]).astype('float32') 

    # Search the FAISS index for the top k
    k = 2
    distances , indices = index.search(query_embediing, k)

    # Get the actual text chunks from our original 'chunks'
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 2 AUGMENT
    # This is the "magic prompt." We combine the retrieved context
    # with the user's query.
    prompt_template = f"""
    Answer the following question using *only* the provided context.
    If the answer is not in the context, say "I don't have that information."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # 3. GENERATE
    # Feed the augmented prompt to the text generation model
    answer = generator(prompt_template,max_new_tokens=256)
    print(f"--- CONTEXT ---\n{context}\n")
    return answer[0]['generated_text']


Loading weights: 100%|██████████| 398/398 [00:29<00:00, 13.40it/s]


In [30]:
# Query 1
import transformers
transformers.logging.set_verbosity_error()

query_1 = "What is the WFH policy?"
print(f"--- QUERY ---\n{query_1}\n")
print(f"Answer: {answer_question(query_1)}\n")

--- QUERY ---
What is the WFH policy?

--- CONTEXT ---
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote

Company Policy Manual:

Answer: 
    Answer the following question using *only* the provided context.
    If the answer is not in the context, say "I don't have that information."

    Context:
    - WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote

Company Policy Manual:

    Question:
    What is the WFH policy?

    Answer:
     All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote.
    I don't have that information. 

    The provided context already contains the answer to the question about the WFH policy. The an

In [31]:
# Query 2

query_2 = "What is the company's dental plan?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")

Query: What is the company's dental plan?
--- CONTEXT ---
Company Policy Manual:

- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote

Answer: 
    Answer the following question using *only* the provided context.
    If the answer is not in the context, say "I don't have that information."

    Context:
    Company Policy Manual:

- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays and Fridays are optional remote

    Question:
    What is the company's dental plan?

    Answer:
     I don't have that information. The provided context does not mention anything about the company's dental plan. It only discusses the WFH policy. Therefore, I cannot determine the details of the dental plan from the given information. The answer is "I don't have that information." I don'